# Live Gesture Generation with Microphone Input

This notebook combines real-time audio processing with the sliding diffusion model to generate gestures from live microphone input.

Note that if the microphone quality or loudness is significantly different than what is in the dataset the model will behave very poorly

In [2]:
# This allows us to import modules from the parent directory
import os
os.chdir("..")

import os

import torch
from torch.amp import autocast
import time
import sounddevice as sd
from IPython.display import clear_output

from model import ContinuousMotionModel
from dataset.dataset import GPUDataset
import utils.utils as utils
import utils.animation.visualisation.new.animation_visualisation as animation_visualisation
from utils.audio_processing.audio_features_extractor import AudioFeaturesExtractor
from utils.performance_tracker import PerformanceTracker

## 1. Configuration and Initialization

In [3]:
# -- Parameters --
MODEL_PATH = "trained_models/pretrained_sliding_diffusion.pth"
SAMPLE_RATE = 16000
FPS = 30
SEQ_LENGTH = 80 # Match the model's expected audio sequence length
AUDIO_AMPLITUDE_CLAMP = 0.1 # Clamp audio amplitude to this value (e.g., 0.8 = 80% volume)

# -- Global Variables --
device = utils.get_device()
live_tracker = PerformanceTracker()
stream = None
feature_extractor = None
# This is an example style, you might want to make this selectable

In [4]:
main_agent_id_one_hot = torch.nn.functional.one_hot(torch.tensor([2]), num_classes=17).float().to(device)

## 2. Load Model and Initialize Services

In [7]:
# Load the gesture generation model
model: ContinuousMotionModel = ContinuousMotionModel.load_model(MODEL_PATH, device)
model.condition_mask_probabilty = 0.0  # Disable condition mask for inference
model = model.to(device)
model.eval() # Set to evaluation mode
print(f"Model '{MODEL_PATH}' loaded with {sum(p.numel() for p in model.parameters())} parameters.")

# Initialize audio feature extractor
feature_extractor = AudioFeaturesExtractor(
    sample_rate=SAMPLE_RATE,
    context_frames=SEQ_LENGTH,
    device=device,
    normalization_file="dataset/genea2023_dataset/trn/main-agent/consolidated_meta.pkl"
)
print("Audio feature extractor initialized.")

# We need a skeleton object for denormalizing poses
# We can get it by creating a dummy dataset instance
dummy_dataset = GPUDataset(
    consolidated_file="dataset/genea2023_dataset/trn/main-agent/consolidated.npz",
    batch_size=1, epoch_length=1, seq_length=SEQ_LENGTH
)
skeleton = dummy_dataset.skeleton
del dummy_dataset

Model 'trained_models/pretrained_sliding_diffusion.pth' loaded with 9169621 parameters.
Audio feature extractor initialized.
Initializing GPU-resident dataset on cuda
Loading data from dataset/genea2023_dataset/trn/main-agent/consolidated.npz directly to GPU...
Audio features with speaking status, shape: torch.Size([2039227, 39])
Finger availability data included, shape: torch.Size([372, 1])
Data loaded to GPU. Gesture shape: torch.Size([2039227, 345]), Audio shape: torch.Size([2039227, 39])
Found 2009839 valid starting points for windows
Dataset initialization complete!


## 3. Setup Real-time Audio Stream

In [8]:
def audio_callback(indata, frames, time_info, status):
    """This function is called for each audio chunk from the microphone."""
    if status:
        print(status)
    audio_chunk = torch.from_numpy(indata[:, 0].copy()).reshape(1, -1).float()

    audio_chunk

    # Clamp the audio to prevent it from being too loud.
    audio_chunk = torch.clamp(audio_chunk, -AUDIO_AMPLITUDE_CLAMP, AUDIO_AMPLITUDE_CLAMP)
    
    if feature_extractor:
        feature_extractor.process_chunk(audio_chunk)

def start_audio_stream():
    """Starts the microphone input stream."""
    global stream
    chunk_size = int(SAMPLE_RATE / FPS)
    stream = sd.InputStream(
        callback=audio_callback,
        samplerate=SAMPLE_RATE,
        channels=1,
        blocksize=chunk_size
    )
    stream.start()
    print("Audio stream started.")

def stop_audio_stream():
    """Stops the microphone input stream."""
    global stream
    if stream:
        stream.stop()
        stream.close()
        stream = None
    print("Audio stream stopped.")

In [9]:
import utils.animation.visualisation.new.animation_visualisation as animation_visualisation
print(animation_visualisation.init_visualization(display=True))

Viewer URL: http://localhost:8001/utils/animation/visualisation/new/animation_viewer.html?wsport=8704


None


In [10]:
try:
    start_audio_stream()
    
    # Initialize gesture sequence with random noise
    encoded_gesture_sequence = torch.randn(
        (1, model.gesture_length, model.pose_features_per_frame),
        device=device, dtype=torch.bfloat16
    )
    # A seed gesture is also needed, initialize with zeros or random
    seed_gesture = torch.zeros(
        (1, model.seed_length, model.pose_features_per_frame),
        device=device, dtype=torch.bfloat16
    )

    print("Starting live inference loop... (Press Ctrl+C to stop)")
    # Give some time for audio buffer to fill up initially
    time.sleep(SEQ_LENGTH / FPS) 

    with autocast(device_type=device.type, dtype=torch.bfloat16):
        with torch.no_grad():
            iteration_counter = 0
            while True:
                frame_start_time = time.time()

                # 1. Get audio features from the live stream
                actual_audio_features = feature_extractor.get_concatenated_features()
                if actual_audio_features is None or actual_audio_features.shape[0] < SEQ_LENGTH:
                    time.sleep(0.01)
                    continue # Wait for enough audio features

                actual_audio_features = actual_audio_features.unsqueeze(0).to(device)

                # 2. Run model inference
                encoded_gesture_sequence, _ = model.inference(
                    encoded_gesture_sequence, 
                    actual_audio_features, 
                    main_agent_id_one_hot, 
                    seed_gesture
                )

                # 3. Decode and denormalize the output frame
                denoised_frame_to_decode = encoded_gesture_sequence[:, model.diffusion.clean_frame_index, :model.original_pose_features_per_frame].unsqueeze(0)
                
                if model.pose_encoder is not None:
                    unencoded_denoised_frame = model.pose_encoder.decode(denoised_frame_to_decode)
                else:
                    unencoded_denoised_frame = denoised_frame_to_decode

                denormalized_frame = skeleton.denormalize_poses(unencoded_denoised_frame).squeeze(0).squeeze(0)

                # 4. Send pose to visualizer
                animation_visualisation.send_pose(denormalized_frame.cpu(), skeleton)
                animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0).to(torch.float32),encoded_gesture_sequence.squeeze(0)[:,:100].to(torch.float32)), dim=1), "full tensor")

                # 5. Performance tracking and output
                frame_time = time.time() - frame_start_time
                live_tracker.record_frame(frame_time)
                iteration_counter += 1

                if iteration_counter % FPS == 0: # Print stats every second
                    clear_output(wait=True)
                    stats = live_tracker.get_stats()
                    # print(f"Iteration: {iteration_counter}")
                    # print(f"Avg FPS: {stats['avg_fps']:.2f}, Frame Time: {stats['avg_frame_time_ms']:.2f} ms")

                # Maintain target FPS
                time_to_sleep = max(0, (1/FPS) - frame_time)
                time.sleep(time_to_sleep)

except KeyboardInterrupt:
    print("\nInference stopped by user.")
finally:
    stop_audio_stream()
    print("Cleaned up resources.")

input overflow

Inference stopped by user.
Audio stream stopped.
Cleaned up resources.
